# CITE-seq: Myeloid Protein Analysis (CLR, SELENOP+/- Comparison)

Analyzes ADT protein expression in CITE-seq myeloid cells:

1. **Load and quality-filter**: keep cells where RNA_annotation == Protein_annotation
2. **CLR normalization** of ADT protein matrix
3. **Panel-wide Kruskal-Wallis + eta-squared** across all cell types (η²≥0.14 cutoff)
4. **Myeloid RNA normalization** → export for AUCell annotation in `01_cite_seq_annotation.R`
5. **SELENOP+/- macrophage comparison**: Mann-Whitney U + Cohen's d, heatmap, violin plot

**Inputs**:
- `RNA_raw_counts.csv`              – cell × gene raw count matrix
- `protein_raw_counts.csv`          – cell × protein raw count matrix
- `annotation_table.csv`            – per-cell RNA / Protein annotations
- `myeloid/myeloid_subtypes_third_round_AUC.csv` – AUCell myeloid subtype labels

**Outputs**:
- `myeloid/norm_counts.csv`          – normalized myeloid RNA (input to `01_cite_seq_annotation.R`)
- `selenop_mye_top_diff_prs.pdf`     – SELENOP+/- protein heatmap
- violin plot: CLR expression of selected proteins by SELENOP+/- group

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.stats as stats
from scipy.stats import gmean, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR = "/path/to/cite-seq/output"
MYE_DIR  = f"{DATA_DIR}/myeloid"

## 1. Load data and quality filter

In [ ]:
rna_counts = pd.read_csv(f"{DATA_DIR}/RNA_raw_counts.csv")
pr_counts  = pd.read_csv(f"{DATA_DIR}/protein_raw_counts.csv")
anno       = pd.read_csv(f"{DATA_DIR}/annotation_table.csv")

# Keep cells where RNA and protein annotations agree
good_cell = [
    cell for cell, rna_ann, prot_ann
    in zip(anno['Unnamed: 0'], anno['RNA_annotation'], anno['Protein_annotation'])
    if rna_ann == prot_ann
]
print(f"Quality-filtered cells: {len(good_cell):,} / {len(anno):,}")

good_anno = anno[anno['Unnamed: 0'].isin(good_cell)]
good_rna  = rna_counts[rna_counts['Unnamed: 0'].isin(good_cell)]
good_pr   = pr_counts[pr_counts['Unnamed: 0'].isin(good_cell)]

print(good_anno['RNA_annotation'].value_counts())

## 2. CLR normalization of ADT protein matrix

In [ ]:
def clr_normalization(adt_df):
    """CLR normalization: log(x / geometric_mean_per_cell) across all proteins."""
    adt_df = adt_df.replace(0, np.nan)
    adt_df = adt_df.fillna(adt_df.min().min() * 0.01)
    geometric_means = gmean(adt_df, axis=1)
    clr_transformed = np.log(adt_df.div(geometric_means, axis=0))
    return pd.DataFrame(clr_transformed, index=adt_df.index, columns=adt_df.columns)

good_pr.index = good_pr['Unnamed: 0'].tolist()
good_pr = good_pr.drop('Unnamed: 0', axis=1)

clr_pr = clr_normalization(good_pr)

# Map broad lineage labels
label_map = {'epithelial': 'Epi', 'granulocyte': 'Granulo',
             'myeloid': 'Myeloid', 'plasma': 'Plasma/B', 'tnk': 'T/NK'}
new_anno = [label_map.get(i, i) for i in good_anno['Protein_annotation'].tolist()]
clr_pr['anno'] = new_anno

## 3. Panel-wide Kruskal-Wallis + eta-squared (across all lineages)

In [ ]:
cell_types = clr_pr['anno']
clr_pr_vals = clr_pr.drop(columns=['anno'])

# Kruskal-Wallis p-values + BH correction
p_values = []
for protein in clr_pr_vals.columns:
    groups = [clr_pr_vals[protein][cell_types == ct].values for ct in cell_types.unique()]
    _, p = stats.kruskal(*groups)
    p_values.append(p)

_, p_adj, _, _ = multipletests(p_values, method='fdr_bh')
results_df = pd.DataFrame({'Protein': clr_pr_vals.columns, 'p_value': p_values, 'p_adj': p_adj})
print(results_df[results_df['p_adj'] < 0.05].sort_values('p_adj'))

In [ ]:
# Eta-squared effect size for Kruskal-Wallis (cutoff ≥ 0.14 = large effect)
# Reference: Sullivan & Feinn (2012) doi:10.1016/j.jpa.2011.11.001
clr_pr['anno'] = new_anno

def eta_squared_kruskal(data, group_col, value_cols):
    results = []
    for protein in value_cols:
        groups = [group[protein].values for _, group in data.groupby(group_col)]
        H_stat, _ = stats.kruskal(*groups)
        N = len(data)
        k = len(groups)
        eta_sq = (H_stat - (k - 1)) / (N - k)
        results.append((protein, eta_sq))
    return pd.DataFrame(results, columns=['Protein', 'Eta_squared']).sort_values('Eta_squared', ascending=False)

protein_cols = clr_pr.columns.difference(['anno'])
eta_sq_results = eta_squared_kruskal(clr_pr, group_col='anno', value_cols=protein_cols)

sig_markers = eta_sq_results[eta_sq_results['Eta_squared'] >= 0.14]
print(f"Proteins with large effect (η²≥0.14): {len(sig_markers)}")
sig_markers

## 4. Myeloid RNA normalization (exported for AUCell annotation)

In [ ]:
# Subset myeloid RNA counts
good_rna['anno'] = good_anno['Protein_annotation'].tolist()
myeloid = good_rna[good_rna['anno'] == 'myeloid'].drop('anno', axis=1)
myeloid.index = myeloid['Unnamed: 0'].tolist()
myeloid = myeloid.drop('Unnamed: 0', axis=1)

# Normalize
myeloid_adata = ad.AnnData(myeloid)
sc.pp.normalize_total(myeloid_adata, inplace=True)
sc.pp.log1p(myeloid_adata)

mye_norm = pd.DataFrame(myeloid_adata.X, index=myeloid_adata.obs_names, columns=myeloid_adata.var_names)
mye_norm.to_csv(f"{MYE_DIR}/norm_counts.csv")
print("Exported myeloid normalized counts for AUCell annotation.")

## 5. SELENOP+/- macrophage protein comparison

Load AUCell subtype labels, subset macrophages (exclude DCs/Neutrophils),
then group into SELENOP+ vs SELENOP- and compare protein expression.

In [ ]:
# Load AUCell myeloid subtype labels (from 01_cite_seq_annotation.R)
aucell_anno = pd.read_csv(f"{MYE_DIR}/myeloid_subtypes_third_round_AUC.csv")

# Attach AUCell labels to myeloid protein matrix
mye_clr_pr = clr_pr[clr_pr['anno'] == 'Myeloid'].copy()
mye_clr_pr['old_anno'] = aucell_anno['label'].tolist()

# Exclude DCs and Neutrophils to focus on macrophages
mac_clr_pr = mye_clr_pr[~mye_clr_pr['old_anno'].isin(['cDC2', 'cDC1', 'CCR7+ DC', 'Neutrophil'])].copy()

# SELENOP+ vs SELENOP- grouping
selenop_pos = {'Mac S+XS-', 'Mac S+SG+', 'Mac S+M+P+', 'Mac S+M+S+'}
mac_clr_pr['selenop_anno'] = [
    'SELENOP+ Mac' if lbl in selenop_pos else 'SELENOP- Mac'
    for lbl in mac_clr_pr['old_anno']
]
print(mac_clr_pr['selenop_anno'].value_counts())

mac_clr_pr_clean = mac_clr_pr.drop('old_anno', axis=1)

In [ ]:
# Mann-Whitney U test + BH correction across all proteins
def mann_whitney_u_test(df, group_col, value_cols):
    """Two-sided Mann-Whitney U test for each protein between two groups."""
    groups = df[group_col].unique()
    if len(groups) != 2:
        raise ValueError("Requires exactly two groups.")
    g1 = df[df[group_col] == groups[0]]
    g2 = df[df[group_col] == groups[1]]
    results = []
    for protein in value_cols:
        stat, p = mannwhitneyu(g1[protein].dropna(), g2[protein].dropna(), alternative='two-sided')
        results.append((protein, p))
    result_df = pd.DataFrame(results, columns=['Protein', 'p_value'])
    result_df['p_value_adj'] = multipletests(result_df['p_value'], method='fdr_bh')[1]
    return result_df.sort_values('p_value_adj')

protein_cols = mac_clr_pr_clean.columns.difference(['selenop_anno', 'anno'])
mann_whitney_results = mann_whitney_u_test(mac_clr_pr_clean, 'selenop_anno', protein_cols)

# Filter FDR < 0.1
mann_whitney_results_sig = mann_whitney_results[mann_whitney_results['p_value_adj'] <= 0.1]
print(f"Significant proteins (FDR<0.1): {len(mann_whitney_results_sig)}")

In [ ]:
# Cohen's d effect size
def cohen_d(g1, g2):
    pooled_std = np.sqrt((np.var(g1, ddof=1) + np.var(g2, ddof=1)) / 2)
    return (np.mean(g1) - np.mean(g2)) / pooled_std

def compute_cohen_d(df, group_col, value_cols):
    """Cohen's d between two groups for each protein."""
    groups = df[group_col].unique()
    g1_data = df[df[group_col] == groups[0]]
    g2_data = df[df[group_col] == groups[1]]
    results = [
        (p, cohen_d(g1_data[p].dropna(), g2_data[p].dropna()))
        for p in value_cols
    ]
    return pd.DataFrame(results, columns=['Protein', 'Cohen_d']).sort_values('Cohen_d', ascending=False)

mac_clr_pr_clean_sig = mac_clr_pr_clean[mann_whitney_results_sig['Protein'].tolist() + ['selenop_anno']]
protein_cols_sig = mac_clr_pr_clean_sig.columns.difference(['selenop_anno'])
cohen_d_results = compute_cohen_d(mac_clr_pr_clean_sig, 'selenop_anno', protein_cols_sig)

# Medium/large effect (|d| >= 0.2); reference: doi:10.3102/10769986006002107
cohen_d_results_effect = cohen_d_results[
    (cohen_d_results['Cohen_d'] >= 0.2) | (cohen_d_results['Cohen_d'] <= -0.2)
].sort_values('Cohen_d', key=abs, ascending=False)
cohen_d_results_effect

## 6. Heatmap: mean CLR protein expression (SELENOP+/-)

In [ ]:
heatmap_data = (
    mac_clr_pr_clean_sig
    .groupby('selenop_anno')
    .mean(numeric_only=True)
    .T
)

# Filter to proteins with meaningful effect size
heatmap_data = heatmap_data[heatmap_data.index.isin(cohen_d_results_effect['Protein'])]

# Clean up protein names (strip antibody prefix)
def clean_protein_name(name):
    for prefix in ['anti-human_', 'anti-mouse-human_', 'anti-human-mouse_']:
        if prefix in name:
            return name.split(prefix)[1]
    return name

heatmap_data.index = [clean_protein_name(i) for i in heatmap_data.index]

plt.figure(figsize=(4, 10))
vmax = heatmap_data.max().max()
sns.heatmap(heatmap_data, cmap='coolwarm', linewidths=0.5, vmin=-vmax, vmax=vmax)
plt.xlabel('Macrophage Group', fontsize=14)
plt.ylabel('Protein', fontsize=14)
plt.title('CLR Protein Expression: SELENOP+/- Macrophages', fontsize=14)
plt.xticks(fontsize=11); plt.yticks(fontsize=11)
plt.tight_layout()
plt.savefig(f"{DATA_DIR}/selenop_mye_top_diff_prs.pdf", format='pdf', bbox_inches='tight')
plt.show()

## 7. Violin plot: selected proteins

In [ ]:
# Selected proteins with strongest differential expression
subset_proteins = [
    'anti-human_CD163', 'anti-human_HLA-DR', 'anti-human_CD52',
    'anti-human_CD39', 'anti-human_CD11a', 'anti-human_CD18',
    'anti-human_CD13', 'anti-human_CD11c'
]

mann_sig_sub  = mann_whitney_results_sig[mann_whitney_results_sig['Protein'].isin(subset_proteins)]
cohen_sub     = cohen_d_results_effect[cohen_d_results_effect['Protein'].isin(subset_proteins)]

mac_plot = mac_clr_pr_clean_sig[subset_proteins + ['selenop_anno']].copy()
mac_plot['anno'] = mac_plot['selenop_anno']

# Sort proteins by effect size (absolute Cohen's d, descending)
sorted_proteins = cohen_sub.sort_values('Cohen_d', key=abs, ascending=False)['Protein'].tolist()[::-1]

df_long = mac_plot.melt(id_vars=['anno'], var_name='Protein', value_name='Expression')
df_long = df_long[df_long['Protein'].isin(sorted_proteins)]
df_long['Protein'] = pd.Categorical(df_long['Protein'], categories=sorted_proteins, ordered=True)

palette = {'SELENOP+ Mac': '#f08080', 'SELENOP- Mac': '#add8e6'}

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.violinplot(x='Protein', y='Expression', hue='anno', data=df_long,
               split=True, palette=palette, bw=0.2, inner=None, ax=ax)

# Annotate Cohen's d
for i, protein in enumerate(sorted_proteins):
    p_adj  = mann_sig_sub.loc[mann_sig_sub['Protein'] == protein, 'p_value_adj'].values
    d_val  = cohen_sub.loc[cohen_sub['Protein'] == protein, 'Cohen_d'].values
    if len(d_val) > 0:
        y_max = df_long.loc[df_long['Protein'] == protein, 'Expression'].max()
        y_range = y_max - df_long['Expression'].min()
        ax.text(i, y_max + y_range * 0.065, f"{-d_val[0]:.2f}", ha='center', va='bottom', fontsize=10)

ax.set_xticklabels([p.split('_')[-1] for p in sorted_proteins], fontsize=10, rotation=0)
ax.set_xlabel('Protein', fontsize=12)
ax.set_ylabel('CLR Expression', fontsize=12)
ax.set_ylim(-9, 9)
ax.set_title('Protein Expression by Macrophage SELENOP Status', fontsize=14)
ax.legend(title='Macrophage Group', loc='lower left', ncol=2)
plt.tight_layout()
plt.show()